# CDCN CVPR 2020 · Kiểm tra MTCNN và PRNet trên OULU-NPU
Dùng runtime **CPU** và chạy các cell theo thứ tự. Mẫu nhỏ lấy 3 video và
9 khung ở mỗi tập Train và Dev của Protocol 1. Mỗi tập có video thật,
ảnh in và phát lại. Kết quả là ảnh scene, bbox, depth JPEG và JSON kiểm tra.
Notebook dành cho bước chuẩn bị dữ liệu trước huấn luyện.

In [ ]:
#@title 1. Kết nối Drive và chọn thư mục OULU-NPU
from pathlib import Path
import os
import shutil
import sys
import time
from google.colab import drive

drive.mount("/content/drive")
DATA = Path("/content/drive/MyDrive/face-pad/datasets/OULU-NPU")
WORK = Path("/content/cdcn-prnet-smoke")
RUN = Path("/content/drive/MyDrive/face-pad/cdcn-prnet-smoke-p1") / str(time.time_ns())
for name, size in {"Protocols.tar": 417792, "Train_files.tar": 37753662976,
                   "Dev_files.tar": 28151992320}.items():
    source = DATA / name
    if not source.is_file() or source.stat().st_size != size:
        raise ValueError(f"Kiểm tra đường dẫn và dung lượng: {source}")
RUN.mkdir(parents=True)
print("Thư mục kết quả:", RUN)

In [ ]:
#@title 2. Nạp code và cài môi trường tách riêng
BUNDLE_SHA256 = '208c4e2062a57bcba7da71c242ded1f0a0e42e521f9b398579922701529f9681'
BUNDLE = (
    'H4sIAAAAAAAC/+08a0/jyJbzOdL9D7WeD2t3OyYPoGk0uRIDdA87/WCBmd0rBlmOXSZuHNtrO0Aml/3tex5VfiQOTO+9Gmm1HalJ'
    'pVx16tR5n1PlLnJ/J5AyCz1fupkX7LhulESl6zrZ8rt/0mcAn/3dXfqGz/r3YPDmTd3G/uF4NB5+Jwbf/QmfRVF6OSz/3f/Pj2EY'
    'J8D+d8B+cX50InJZSC/3ZyLz/DvvVjow4C+9v/Rc917mRZQmrismwhg4Q2cAD7779vm//SnW9T/L01sQguLP0//9/fHuuv6Pd8ff'
    '9P9P0v9zmUdpEPlCs16EaS6StJTTNL0TX9JpIbwkEHGaZgXag16Yp3PhuuGiXOQSDEI0z9K8hEEwySvBSBQ9HuOnSSkfyzia6jHF'
    'IstwjZ76nRY8MvPKWWPYOfzUQ4roNvHi6tdiCnj6DRDlLJdeECW3VUc0l71ez4892Mu52tRhT8AnkKHQHs4sZBzaIvamMrZFCajH'
    'k09pIm2xgOcTIyrlvDBsEcEe8nt4OB5YDAU/ONlRc6lNAFQbAaimng1Ws7kSL1IDb8MNAA2YMFhfrSjdRRZ4JT6kLhTeUgbwEzft'
    'zFPgQJpEvmm1pxZlmuEgTSrn9F4mJYyqqMJwFU1w/cZeo5B6xL9MavTqp+tY49fmwzbyG9hWeMyBV+B3FCKwv7KJSZI+PLdXGXtZ'
    'QeTAgf0WjapBCs0K5eqBny6SEp6EhljRFu2nnVXNW/gpVhV7nwwkS/1URAXqjEAJAjwKKUBRKrxKD72m0aQobU1MoDtfJAnwxCAt'
    'a0DEnwPxAyP8Q+NJm/ZREMv2jhu0bo1kNEzY39/F6dWR+O+V2Viuz2wXryoy7lDPoTMInwraLi31w5pgvxKj1ioVL4AGtFSSikQ+'
    'KPKibVkhHAW2Zl4uwZwkMOV6VSvX0w3QHCn1tKL5TwBPo7dSDYa0gt09GbUguTMIZMqp9EraZEOIHmYRbAOZVemG8+BFPK7al9Wm'
    'cpbDAx6hRbRinGWLMF4Us8lVvpANWXZdicCUqWkA7AKmxHQdWEuJWH9binxFLRMm38pyQqPqnYMmexIUZTss1g5zgws4pLWRx4bJ'
    'lI++Wy4zyS2g1QKaZQ4hxBSixnUjSfQtZGl2I/AljZLGoy7ShB4wLCAJ1EsrFQvBlBezTar1eog28MeN09tbGZh+Op+DPtniVWXx'
    '4Ykt/IdAGX2Z3KtWITMvB5FzlaNxb/N0kU3eebCk2hx4wXdp/uDlAYj1fBolII4+iFUg0kWZLcAHpcAjKXwZsx6j0wOxktSb5tFt'
    'BD5NyDxPc3KpCBQQAtai6zOhaek+B7ABOXLmd0GUm/yjoH0i/SNQ9fSuwWC1U4B0XZS5CXJhkdJBA7yNfnzTYz507ZSMY+cDhJsW'
    'TuLN2XJlaRE9MvLMt9D4kN4eihWg/WRsCvJDVM5oS2kmE9N4NJDsfoqCPDEWZdg/MCzhFUhEu/LcpuJX5Yf3m36YQNZBgXNOoCt2'
    'I3/hH7MX/qFLCQD8pDnl7PzU7jJhTR9WBsCs5qzLq5PPv1yB3EOAo7mxthebGVxMjFxmMaiH8fIyoI0uWEu3gDUgkpp0c4KohD1t'
    'I1Xmy8ONFZD5MUgoch+nOEyBw05U4IHzkEPoY+IUa+sY4qzZ/ZxFAecjSYKJYXSbtNrvBujACDeyw+0hoPIyK8WPXiFPqQlk2USe'
    'nHEXqbr3qcSGg1HznCd8gHh3kZ0iz6zDrZwCBbiL4ji7NQnjLALp4hjVuTx7f3V68XFzizIG/Hh4GsemhdHCp40oqiYgDARhn4OR'
    'AE5sguvkczWTaIgxEsr5XgcyTNCGLF/x4NPHLMrBn3aC/koC/4NEfpnQP599+NAtf+gatkMmWAh3q/R2iyE5Ry8qZDOIQ9FtL0VD'
    'xAWEKkBT2iUYxRXHMmiuMVonupDUr/DvkyPeLcBPsOvQ1tP6U4orhZ9HWVns+IGfAFsTWbrFPL2T/8T0/4X8fzwc7e9t5P/j0bf8'
    '/0/K/y8kpOkQKiz8UhyfHH/qF+US4uOPV8efPu2cX3ySJXgOjGkgnZYlVwc8MYVoPABZJmkBy1D+4cqA7stvwZqAOqnffnGvmzOv'
    'wGKA/slfXR0QIpYepDkeLwzKlZdpGhd6XX/mRYme9aVIq/a8UV94pgQBQQjs4xyjnFZFYrYoo2cqEsWyLk54eQjBa/VTzrPWbyxV'
    '6DaEtTV5/PuRbiaLebZEh59kjGnhR9nSKQDhCBNPHvXvM7AhZHBU8aVZ1nOq2o7enfrd6wGHT6/c488fP55dYZIa+sNRKPfkMBzu'
    '7o+mgfT2RruDt9PRcBwM/HDoH/jhvr83NXonp+dXPzVm+vvBcBCO5HTfm77dH3nT8XA3PBhKb9fzBgMvHAZvx8F0CjP/4/Ts/U9X'
    '7uVPRzhv720wHh9Mw4E/Gk6Hg6k8GI7D4ZvwYHewO/Lhobc3DcfDA7l3cHCwtxeGo+nIH+wO/dHedPp26Bm9s08np/+pwfnTg4Ph'
    '21Hoh292d6cHcgRoSW/fG+0fDP3g7SAcDkbeKHj75o03HULCMh6M9qdBuOvLNwd7MMXoXX78/POpe350/PPR+9NLTJk56C9lUqR5'
    'GKcPk8nIGe47IxXQGRDu3cncK1T/oNFfxNF8MhlSwVz1zks/SbBvUPcRk7FvtO/s6k5g9F1U9qM5pEGTycAZ7dYTSAgI8BtnqDsx'
    '+PXv+9mynKVJH/JASLQLwGrXGeJqB/sw0FL7u/zw+Yo2N7DF2BZvrN75GXRV/eUii6WZewmkYAeWzqmKmWeipijfTa4Mf6uQPp9y'
    'BA/2RHrzw/W0Umm2gzrgBtGtLCDXo6E27Gjmjfb2DcuZyUf1TK8ap17gzsFXxoVJXsoGiGB/cqmDCAgWILyP8jS5Nq7eub9cnrof'
    'Tt8fHf/N/fn04ujSuEHRGBobY49/OTlyfz27PPvxw6l7cvrr2fGpGsxjK8XVrHcgt4ANO/dD3GcZ0ihSOGKrnkDmkzPo78XVO4Fy'
    '8a+F4MnRNIqjcknbkjmEBZn0wbJSbgg5pDp7gcc+GU2Hk7Vl4QAJFsDRaxAslwTOieWt5y9d+B17S5kXhHttILnl8jyznlbk/uZU'
    '6yvWcZI0n3tx9Dth+OyqlRA8v/waRJplqRJECfQBtzNhspqBvI98OTGOz385HGi0OWGCmNTLZii+ofMem2Yj9WyNcbzCBdnyFjHI'
    'WS2p34sPwBXiRRYlmNhD+BpEtD4B8YCdw36U9K/ejdb4yft26mIGKAZlNiCxYgfyZQ0Jwiujt57/oBZhScTFlNLcSIwdlUSarXjT'
    '2BBQFsvfki7JTcoc2MP0pnF+/FtCwtsxLMwhz39I8ztRu2y38EHR1/JY40UlqbEJXTSIa2urzmeWqXePltLlULnYAQMsWh3Q06wl'
    'acZN2Ls6H0kir6Bt1sxozGjJfj3ghpiofrUGE9OipIDAA+0oVlyI21YNcyNZk4/SxwpFBkbQROareWhVYTv4GOtZtbC4LjZct52Q'
    'EOm8LKpc+sWn1nMyl1hQuvhkRiDqoJhcwyLQYfQ46cI2xMJUvIZxtdMszcyBtfFMkYweN8hmU47Lw9lcdzIigGx05pKTU6xQeqNN'
    'PKoORlzFTmNopUItctZKNF2WsjCtbtoy5DXCKh+lrY3NFKwG38pEUvbbQEIXGusQ2iStsbdBUYYmv53C/iDGc/z78jiNMUvkedh3'
    '/PnD5wv3x/cXo4v3PzJuGMgVeLiiwDrccKnfBHA8DFJSrGtTb8P5Uk76K9ZpOSM1yIyKIApoOMMCw4bzFAsebbG0xYMtkBFz79Ek'
    'mLa4k8tJ7M2ngYfh9vyQ/l4b0/TRuLke3YhXrY7xjaWazFeIpCkLQJhIFkdFzrgDG+Jbx8tzb2lew+qP4jUuD0gsoTW7UTJ6D5oW'
    'MS0Ywi2krLrTrBbQjiNrCJKph9mNqbGXBHMvv2vOtZE/DoQjWOKOio8yeBfFkFs06ldKVsyaTJbNyymR8PM0cylJKkwggC0UuML3'
    'YglR21hJQovOMJA6ZzK6nUGM8xAFhD/NvT4c3ah44mcI7clDcQABcQVYy5z8pkhz1DMPhAAeluBA+lkaJSU4s0cwB4SRl0egglwt'
    '5hMKwEK1lhgRIu3pz4NlgfKNbGEiE+jPjLsYTy8OXcQdvwlR3B3IwAPP0j9n1YxYhqUSKDCXmIY5gCeIpEZE9BVUxe8cKYEzosQk'
    'cnTPet2excedW1dZ6lVmavw0LUuwpryMpn7XvNfteaBvtKO/ThSi4Gtw7b9OFMjnlBByQYgZiEYBqR7y32jJF7hFcHkA0Vbw0J5R'
    'Hy5r86JVaC5jNAn3USBTDJPTMvVTPJLO4oiFSbqFN4eovnWWQUGNylM5iq+nGvkhx/N4Jyi6l83t4Ems6nYgYMk9v0QQZmicq/nF'
    'jm65w50VofHklI8lxTJeYFpgxdD/mRzi9IvoVm8/fUANvzapkozFM9g4Hjs4YGWjzLT4TEOdDNB5BJ5tFPcEGFQd0HNoQSxDFzge'
    'WYXDNQStS58TKYhkBFF6ePcJkie5o4JFdcDDyoUHIDSEaY2nPnjLgHWJpunCPaM/tMUQMDd3bTGybhCHBhf4DEsNYnT8WaqM27VJ'
    '4Gwds9rC89FS8sbbz6iy316cijQ0AR/iEqCOu2oVlhBYZPXU0ycE3asRVRmnmvUdlEd+tUOGMJJxQBtBFqJPYMyxhTOYmcgh03AN'
    'hVijsmoygOvBja1gXQ/rJvgVPIDaRqPO4wGWI5w2tPAqg6m2SB3d9eINnQ0NBjKPCrAO/uxQrHAjT8ZmrZiJfI2PMXzkxTZGTUFe'
    '73rbK9cdGMDaBZh1XXkDxSsX4IeVfokV0YSuKhBRoMVrP7VNC+OnTIecR8pwUKikCVnZDg6vld/E7jgti0mjjkAHq0XG6QsgnbRM'
    'jO9lKtz5Fdc49jKsD5pVZKaMdjNQ1pcxUHpgOvpqk2Kjo3P3/OLzufvu4ujjqXv8+ZdPV43wlWryOPMH4DeYMkKUlL+JHGvecGC9'
    'SGyfSpcUIeFxjBQUpvEaQHMKN9clAJXBFgu+iHJ9g+LZPEz4XpzIe4ijZMYZf7aYgkWvXTnap0A64oowBQSEMhcexBQFz3mYpdBx'
    '9OuZ09JK3C0qF+26vbckndORMwg7ODEa+EqRamdHvEiqeo5ZTRquZwLK64IL1au9FnvggHnGa1CzdS1vrrSpggogLk7YEYhutDcP'
    'FcFUR3gDBm0Q1YFVKUuhZhN0DPWotyAfi6jTout4Wy8d3Cr5aSwAIcbQ1iFI3beH/qA/tNbOODuPrFp76MJ+7fA3IolrS5sWDZJb'
    'FwJ+likFt9NQNoeiFHfbRqwSRMlCvgSB0PpKEDTH8YLA1JC6GMy3SlpG4fzzJRsGMEgVEv11waMa4J0ageIBoDgg6doM6n96h5Gd'
    'Ny26bVG9rAWrVThjGDhw9r5681sOdymNqFOajZTzFVvn7mPNPI1wp+tZCU3lVGfrQbFZW0S7ccxgfb1cEPK8GnrgxuKY3CCJeURA'
    'V3xgBGSEC/AAB3/QPRtcSQU1gFwW8OwXELhLQSAU5eZeZnRLAxXcZODOK2cVzbnPNJwv2a2hiP+MkDyH5udLhSNYlZjzboauEPu3'
    '89P3HZgBNuORwieXRfS7NBk1FTo3sObiwdnHi9OjE/f9xdHfLo+PPpxa18B6cELmGCLA8cjaij6t5HjJ0rSQE83wiOJeHAP8yMrM'
    'pKHW1wrA9+IjxkzkwNRZ4ob/A7nGRiGmEqyWFLRlvOfW4RzknC5rshPGn0/uSuve4WAcPBmbk1AeXFVeUle1djjIwSIT1XQKA5oV'
    'TGjDEgydZpModLFpO1h4uB0ocX940g22xvcrb4F14fYPgFAiwpLH94MweKvRs5Qpsf6ICoQNHSBgEKxWkJ6MZ3ZAo1VpryH5Tpmq'
    'cp/1HAkx03WLRRhGj6DS4AAhEWWIVHE3fksMvo4IIHF/mHQ/Qur4WuCjTdDoHF/wUhgLOl4GWXVgrgyKsGGccVjLAh58oaBAnwq2'
    'DdY86NBRuIFBDg6ArxciEUPjAsN1E2vkHDdAp44gXgCj7oIs5u54BLMwDGcDAV1AZgBJQZvLF0/d32WeVgij5R4+WS8kOyoPQQr1'
    'OivP7JZj6RWyOgicYyCnhIyuD+RUheCrBM5RfrsAm12e0xMzkHy9Be/RuW6Q+rrSyzORda6npphGv483CcDKo/OZ4Hn/s4O5Xv1H'
    'R2Pa3+c43sDsinAyIIjLQfhA5aBzJuNsYkDWlOqAnzI5sVvn+SPj2UWII1Txi1Benl0HAm+0AIJqXSwphciAnCSjz69DstHcOla3'
    '/2uBd8ca9mPLZF3U/9/OZ/P64myYUtChGgGhLwRTmFXhDq/GFmgFPH0bj2sK1E3crfobN7cZnmRXjnfb4iXQEPxYLliA0Hdq6ahK'
    '1roi1Ci43MklIp6ltqiiFizVmIYmLyGiTrubtzSwoGQ0yEjj9G9bNK9lNH003omNMKVuXPzzZ9K/U4dn5rVxG+G6Rv/Y4GMTRA9V'
    'PZf3fdo5Pv0JwgvjpnHx1dIltXYWTquBJdC7ezHXXgFJngShBBiJ+aIoIQwQKw2g6Rk0Ta9hDpZXeDmuZ0pVMq7ph074BHizA21i'
    '0s5ob9/FfxBfhH6CLZ5G8tCne2j9NKSGujQQ8d0HHkYFpMYtFsy9q4drfiYCxjwaFk2pLqo8VxdWARFfvSJycAUfpvUBUxFEYVgf'
    '19MumYPPe3YuyNQl3+p+tUEH3nwkhIEeLWw0JIfzCnyDqXENoymdLQmsFKxWpMarRKRXOrympEvdUldD27Wclhr+8WIN362sgTYl'
    'Z2sWRSM7sigV+3BmQuHxC4gYzDn2b5iLJIJL/e3cYz2YajAS5ZXXy/j1krXEg29YG2wUzj4evcfbSpeXhs27w4c8HcsP6KyxB2J3'
    'Kmi4SZqgu+Yh1pb3TRj7Xl3PwpcwFjnXpkH7r9UbBFh9wUPmxtUhzXrtjlxyR1xIapQMe9WF9+0vsFFRi8JoNI0GlcTQBp3I+6aA'
    'VgXt9hFIbeCBntWhhANLGlVlk8Y0ivI1BdTZhg7rm8AweKdqq4snHgxxjTnVkEN8cfKOdnL065mYy/kUL1/AflbNFVQVsZsZbX2t'
    'QIurowvSWOIkzMV6GO8drTa3DP2uAE22xW+bBYfm2U8Tp83zH7u6wehcSbx24OXLE/C8eCSNeSNe64D+NQWRcxYEVZxSGG6UqJg0'
    'SJmN06aqiskFm9atTJPnEQG3J7c8JiroiIoSXD6BICvtxOkDxItkog3Hu4+MegCmawmXVauNfGXii/e4Yv1qD9IHT1AJOv7pdV6b'
    '7zpZ403w/bp0kfvSZtDq7t2DunsHYe+Wtzv44iqYgWyJANPpF1MDwkm22BWvxHAw0l8dhZIpvsbk6UP8xrkBYWI3jz2IdjdtTes8'
    'SNiSNsHWMW1qLml1E2vddnI46+CN3xhyPCaPt+1toy9AgmT9rcqmYDYwQDFoIrT9PQMFVb1Ng6g4wWKeFc39bE8t6WA5ae8dpbOu'
    'eG9BV1npzpyzYoqtSQSdG8t0JG4sZYskjpK7rtcitF5ADDBP76VZLbQ5tvr/BdQLxw2TJfqETAXN6pq9ZltXz81/2mk+ftJu4u9i'
    'te5s+soPHTpDfCn1+feVlEl5wRq0k90Wb5Rgt3hDTbyhhCEE/K6O+WqnYTyRNPJhNZ7ypIjyBr3U7VFy1AVEzvqUdTIxrOvBjfKp'
    '0kcQ7ZvON+p8kK6TQbJSCwmHAIbeAdYWVBOLE2RCcA8qKIc+joRddaX3sBErrxce9FAKk+sJVaRsYy1C3SKonmKw/Yx3tzYXqT08'
    'linWnL6qsJA6RHglmTRsA4imrKGOfQ873kRwFBFIB/jIWx+S6+lPG4ALDEMDhLsumPhKN4smZ47rlo4OgbFMMwdPTAavXdNq2B3m'
    'K8aCAaQHk1F9hUXz8pmkJDQu6bg5SvCmXSxL2PxKz9PhtQpL8SUSl2NTLV8Um5JygiBZtflZf2s3wneNkV7438yAL3ZdlG3XNRg1'
    'rv58+19nvn2+fb59vn2+fb59/rHP/wDXNk7PAFAAAA=='
)
import base64
import hashlib
import io
import tarfile
bundle = base64.b64decode(BUNDLE)
if hashlib.sha256(bundle).hexdigest() != BUNDLE_SHA256:
    raise ValueError("Code đi kèm không khớp SHA-256")
WORK.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(bundle), mode="r:gz") as archive:
    for member in archive:
        dest = (WORK / member.name).resolve()
        if not member.isfile() or not dest.is_relative_to(WORK):
            raise ValueError(f"Invalid bundle member: {member.name}")
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(archive.extractfile(member).read())
(RUN / f"source-{BUNDLE_SHA256[:12]}.tar.gz").write_bytes(bundle)
sys.path.insert(0, str(WORK / "src"))
from deepface_pad.progress import Progress, run_logged

def run(label, *args):
    run_logged(args, label=label, log=RUN / f"{label}-{time.time_ns()}.log",
               env={**os.environ, "PYTHONPATH": str(WORK / "src"), "TF_ENABLE_ONEDNN_OPTS": "0"})

run("install-uv", sys.executable, "-m", "pip", "install", "uv")
VENV = WORK / "venv"
PY = VENV / "bin/python"
if not PY.is_file():
    run("create-venv", sys.executable, "-m", "uv", "venv", "--python", "3.12.10", VENV)
run("install-prnet", sys.executable, "-m", "uv", "pip", "install", "--python", PY,
    *('tensorflow==2.16.2', 'tf-keras==2.16.0', 'tf-slim==1.1.0', 'mtcnn==1.0.0', 'numpy==1.26.4', 'scikit-image==0.24.0', 'scipy==1.17.1', 'opencv-python-headless==4.11.0.86'))

In [ ]:
#@title 3. Tải code và trọng số PRNet công khai
PRNET = WORK / "PRNet"
RENDERER = WORK / "PRNet-Depth-Generation"
for path, repo, commit in [
    (PRNET, "https://github.com/yfeng95/PRNet.git", "fc12fe5e1f1462bdea52409b213d0cf1c8cf6c5b"),
    (RENDERER, "https://github.com/clks-wzz/PRNet-Depth-Generation.git", "c6d10f2eb6ab962ab314f81ea4aa00af1d93dbbb"),
]:
    if not path.exists():
        run(f"clone-{path.name}", "git", "clone", repo, path)
    run(f"checkout-{path.name}", "git", "-C", path, "checkout", "--detach", commit)
WEIGHT = PRNET / "Data/net-data/256_256_resfcn256_weight.data-00000-of-00001"
def weight_matches(path):
    if not path.is_file():
        return False
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest() == "59d338bf0c21b10be813f17f84042c9d3a5bf318e588855ff2b2c041c25bb91a"

if not weight_matches(WEIGHT):
    partial = WEIGHT.with_suffix(".part")
    run("download-prnet", "curl", "-LfsS", "--retry", "2", "-o", partial,
        "https://drive.usercontent.google.com/download?id=1UoE-XuW1SDLUjZmJPkIZ1MLxvQFgmTFH&export=download&confirm=t")
    if not weight_matches(partial):
        raise ValueError("Tệp tải về không khớp SHA-256 của checkpoint PRNet")
    partial.replace(WEIGHT)
run("verify-prnet", PY, "-u", WORK / "scripts/cdcn_prnet_smoke.py",
    "--image", PRNET / "TestImages/0.jpg", "--prnet", PRNET, "--renderer", RENDERER,
    "--output", WORK / f"public-{time.time_ns()}")

In [ ]:
#@title 4. Dựng dữ liệu mẫu Train/Dev và lưu lên Drive
OUT = WORK / f"oulu-{time.time_ns()}"
WIDE_SAMPLE = False  # True để thêm phone 4/session 2, đọc sâu hơn trong TAR
run("oulu-prnet-smoke", PY, "-u", WORK / "scripts/cdcn_prnet_smoke.py",
    "--data", DATA, "--prnet", PRNET, "--renderer", RENDERER, "--output", OUT,
    *(["--wide-sample"] if WIDE_SAMPLE else []))
with Progress("Save smoke assets to Drive"):
    local_zip = Path(shutil.make_archive(str(OUT), "zip", OUT))
    temporary = RUN / (local_zip.name + ".part")
    shutil.copyfile(local_zip, temporary)
    temporary.replace(RUN / local_zip.name)
    shutil.copyfile(OUT / "smoke_summary.json", RUN / "smoke_summary.json")
print("CDCN_PRNET_SMOKE_READY", RUN)